In [19]:
from pathlib import Path
import sys
import pandas as pd

In [20]:
ROOT   = Path("data/processed/schedule")
DEST   = Path("data/feature_store/sos_v0.parquet")
HCA_PTS = 3.0

In [21]:
def load_one(season_dir: Path) -> pd.DataFrame:
    f = season_dir / "league_regular_season_schedule.csv"
    df = pd.read_csv(f)
    df["season"] = season_dir.name               # e.g. 2023-24
    # ── schema normalisation ──────────────────────────────────────────────
    rename = {
        "pf"           : "pts_for",
        "opp_pf"       : "pts_against",
        "home_team"         : "home",
        "away_team"         : "away",
        "team"         : "team_id",
    }
    df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns})
    req = {"season", "team_id", "opponent_id", "pts_for", "pts_against", "home"}
    missing = req.difference(df.columns)
    if missing:
        raise ValueError(f"{f}: missing cols {sorted(missing)}")
    # ensure ints (CSV may load as float 120.0)
    for col in ["pts_for", "pts_against", "team_id", "opponent_id", "home"]:
        df[col] = df[col].astype(int)
    # MOV adjusted
    df["mov_adj"] = df["differential"] - HCA_PTS * df["home"]
    return df[list(req) + ["mov_adj"]]

In [22]:
def load_all() -> pd.DataFrame:
    frames = [load_one(sd) for sd in ROOT.iterdir() if sd.is_dir()]
    return pd.concat(frames, ignore_index=True)


In [23]:
def sos_prev_season(df: pd.DataFrame) -> pd.DataFrame:
    seasons = sorted(df["season"].unique())
    out = []
    for prev, curr in zip(seasons[:-1], seasons[1:]):
        mov_prev = (df.query("season == @prev")
                      .groupby("team_id")["mov_adj"]
                      .mean()
                      .rename("mov_prev"))
        tmp = (df.query("season == @curr")[["team_id", "opponent_id"]]
                 .merge(mov_prev, left_on="opponent_id", right_index=True))
        sos = (tmp.groupby("team_id")["mov_prev"]
                    .mean()
                    .rename("sos_v0")
                    .reset_index())
        sos["season"] = curr
        out.append(sos)
    return pd.concat(out, ignore_index=True)

In [24]:
def main() -> None:
    if not ROOT.exists():
        raise FileNotFoundError(f"Input directory does not exist: {ROOT}")
    df  = load_all()
    sos = sos_prev_season(df)
    DEST.parent.mkdir(parents=True, exist_ok=True)
    sos.to_parquet(DEST, index=False)
    print(f"✅ wrote {len(sos)} rows → {DEST}")

if __name__ == "__main__":
    main()

FileNotFoundError: Input directory does not exist: data\processed\schedule